###Build Drivers Dimension
1. Read silver drivers table
2. Read silver ref_nationality_region table
3. Join the data from drivers with ref_nationality_region using nationality
4. Select the required columns
5. Write the transformed data to gold dim_drivers table

In [0]:
%run ../00-Common/01.environment-config

In [0]:
from pyspark.sql import functions as f

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
drivers_df  = spark.table(f"{catalog_name}.{silver_schema}.drivers")
ref_nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
dim_drivers_df = (
    drivers_df
        .join(
            ref_nationality_region_df,
            drivers_df.nationality == ref_nationality_region_df.nationality,
            "left"
            )
        .select(
            drivers_df.driver_id,
            drivers_df.driver_name,
            drivers_df.date_of_birth,
            drivers_df.nationality,
            ref_nationality_region_df.region.alias("nationality_region")
        )
)

In [0]:
display(dim_drivers_df)

In [0]:
(
    dim_drivers_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))